# ДЗ 2. Поиск утечек данных

Предсказываем опоздание заказа в момент его оформления.
Таргет: late_delivery, где 1 — опоздал, 0 — доставлен вовремя.

Нужно найти три утечки, объяснить их происхождение и сравнить
ROC-AUC до и после удаления.

Во всех экспериментах:
- HistGradientBoostingClassifier(random_state=42), остальные параметры по умолчанию;
- одно случайное разбиение 75/25 со стратификацией;
- ROC-AUC по вероятностям класса 1;
- без подбора параметров и создания новых признаков.

In [1]:
import sys
import numpy as np
import pandas as pd
import sklearn

from IPython.display import display
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

SEED = 42

df = pd.read_csv(
    "data/olist_delivery_export.csv",
    parse_dates=["ordered_at"]
)

category_columns = [
    "client_state",
    "partner_state",
    "category",
    "pay_method"
]

for column in category_columns:
    df[column] = df[column].astype("category")

print("Размер:", df.shape)
display(df.head())
display(df.isna().sum())
display(df["late_delivery"].value_counts())

Размер: (96470, 29)


,ordered_at,client_key,partner_id,client_state,partner_state,intra_state,dist_km,promised_days,dispatch_deadline_days,items_qty,...,installments,paid_value,late_delivery,order_hour,order_weekday,photos_qty,title_len,handover_lag_days,seller_score,record_id
0,2018-03-03 17:39:37,568843f1b07989b16ad67462e4f7eddf,0bae85eb84b9fb3bd773911e89288d54,SC,SP,0,82.424278,11.264155,5.007500,1,...,1.0,49.17,0,17,5,4.0,47.0,1.977211,0.9923,125097
1,2018-07-15 22:36:39,7fe36cf7ee4c8a68c57dbf47d31c952c,87d3c3aeb3ead335511b3ce315eb341e,PI,SP,0,2093.948450,16.057882,4.009479,1,...,1.0,127.46,0,22,6,1.0,28.0,0.604410,0.9333,108873
2,2017-11-25 00:55:18,3e7688aca4163064b392dba8b3184807,e5a3438891c0bfdb9394643f95273d8e,SP,SP,1,166.125935,15.961597,9.471771,1,...,1.0,35.41,0,0,5,NaN,NaN,9.846748,0.8122,124268
3,2017-05-21 23:24:21,9bab4772c790be2269300fe0d3a0ed69,5cf13accae3222c70a9cac40818ae839,RJ,MG,0,568.116589,22.024757,7.170949,1,...,1.0,95.01,0,23,6,2.0,51.0,1.482477,0.9660,139775
4,2018-08-06 09:09:46,839a04051a2c347397fd425d01fc3eb2,1fe5540d7c1c37a595fefbacd5570d9e,PR,RJ,0,1204.797590,35.618218,8.010266,2,...,2.0,378.38,0,9,0,1.0,51.0,1.205718,0.9500,130113


ordered_at                   0
client_key                   0
partner_id                   0
client_state                 0
partner_state                0
intra_state                  0
dist_km                    476
promised_days                0
dispatch_deadline_days       0
items_qty                    0
partners_qty                 0
skus_qty                     0
goods_value                  0
freight_value                0
freight_ratio                0
category                  1358
weight_g                    16
volume_cm3                  16
pay_method                   1
installments                 1
paid_value                   1
late_delivery                0
order_hour                   0
order_weekday                0
photos_qty                1358
title_len                 1358
handover_lag_days            1
seller_score                 0
record_id                    0
dtype: int64

late_delivery
0    88644
1     7826
Name: count, dtype: int64

In [2]:
excluded_columns = [
    "ordered_at",
    "client_key",
    "partner_id",
    "late_delivery"
]

X = df.drop(columns=excluded_columns)
y = df["late_delivery"]

all_features = X.columns.tolist()

train_idx, test_idx = train_test_split(
    df.index,
    test_size=0.25,
    stratify=y,
    random_state=SEED
)

display(pd.DataFrame({
    "rows": [len(train_idx), len(test_idx)],
    "late_share": [
        y.loc[train_idx].mean(),
        y.loc[test_idx].mean()
    ]
}, index=["train", "test"]))

print("Признаков:", len(all_features))

,rows,late_share
train,72352,0.081117
test,24118,0.081143


Признаков: 25


In [3]:
def evaluate_model(columns):
    model = HistGradientBoostingClassifier(random_state=SEED)

    model.fit(
        X.loc[train_idx, columns],
        y.loc[train_idx]
    )

    probabilities = model.predict_proba(
        X.loc[test_idx, columns]
    )[:, 1]

    return roc_auc_score(y.loc[test_idx], probabilities)


baseline_auc = evaluate_model(all_features)
print("Исходный ROC-AUC:", round(baseline_auc, 6))

Исходный ROC-AUC: 0.911209
